# SNOTEL combo plot
---
Plot snow disappearance dates of input datasets against corresponding SNOTEL dates.  
Abbreviated bar plot.  

*J. Michelle Hu  
University of Utah  
May 2025* 

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import PurePath
from typing import List

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
import seaborn as sns
cmap = sns.color_palette('icefire')
cmap

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# basins = ['blue', 'animas', 'yampa']
basins = ['yampa', 'animas', 'blue']
WYs = [2021, 2022, 2023, 2024]

# Extract peak from iSnobal outputs

In [ ]:
def locate_snotel_extracts(basins: List[str], WYs: List[int], modeldir: str) -> List[str]:
    """
    Locate the SNOTEL extracts for the given basins and water years.
    Parameters
    ----------
    basins : list of str
        List of basin names to locate SNOTEL extracts for.
    WYs : list of int
        List of water years to locate SNOTEL extracts for.
    Returns
    -------
    list of str
        List of file paths to the SNOTEL extracts.
    """
    # Match selected basin and allowed water year
    isnobal_csvs = [h.fn_list(modeldir, f'{basin}*/wy{WY}/{basin}*thickness_snotelmetloom*.csv') for basin in basins for WY in WYs]
    # Flatten into a single list
    isnobal_csvs = [item for sublist in isnobal_csvs for item in sublist]
    return isnobal_csvs
# Thickness SNOTEL data is in each wydir as yampa_HRRR-MODIS_thickness_snotelmetloom_wy2021.csv oryampa_iSnobal-HRRR_thickness_snotelmetloom_wy2021.csv
# TODO: Need to update the HRRR-MODIS to HRRR-SPIReS in the extract code (separate from this)
modeldir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs'

isnobal_csvs = locate_snotel_extracts(basins=basins, WYs=WYs, modeldir=modeldir)
isnobal_csvs

In [ ]:
def extract_peak(fp: str, compute_doy: bool = True, verbose: bool = True):
    # Note basin, implementation, and WY, which you can pull directly from the filename, this will be used for the column names
    basin, run, snowvar, _, WY = PurePath(fp).stem.split('_')
    # Handle run names
    if run == 'HRRR-MODIS':
        run = 'HRRR-SPIReS'
    elif run == 'iSnobal-HRRR':
        run = 'Baseline'
    else:
        raise ValueError(f'Unknown run name: {run}')
    WY = WY.split('wy')[1]
    if verbose:
        print(basin, run, snowvar, WY)

    # Load the data for this file
    df = pd.read_csv(fp, index_col=0, parse_dates=True)
    # Extract SNOTEL sitenames from columns
    sitenames = df.columns
    # Reformat into file-compatible column names
    sitenames = [s.replace(' ', '').replace('-', '_').replace('#', '').replace('(', '_').replace(')', '') for s in sitenames]

    df_fullnames = [f'{basin}_{run}_{WY}_{site}' for site in sitenames]
    df_rownames = [f'{basin}_{site}' for site in sitenames]
    df_colname = f'{run}_wy{WY}'
    # Compute the peak for each series and only grab the timestamp, not the full series
    peak_list = [proc.calc_peak(snow_property=df[dfcol], verbose=False)[0] for dfcol in df.columns]
    if verbose:
        print(peak_list)

    if compute_doy:
        # Convert these timestamps to just that day of year (cut off the hours)
        doy_peak_list = [ts.dayofyear for ts in peak_list]
        if verbose:
            print(doy_peak_list)

    return df_colname, df_rownames, df_fullnames, sitenames, peak_list

In [ ]:
def construct_peak_df(isnobal_csvs):
    # Load data calculated peak and store in dict
    peak_dict = dict()
    for fp in isnobal_csvs:
        df_colname, df_rownames, df_fullnames, sitenames, peak_list = extract_peak(fp=fp, compute_doy=False, verbose=False)
        peak_dict[PurePath(fp).stem] = df_colname, df_rownames, df_fullnames, peak_list

    # # Value for first key in dict
    # peak_dict[list(peak_dict.keys())[0]]

    # Extract relevant data from the dict to construt a dataframe
    colnames = [peak_dict[k][0] for k in list(peak_dict.keys())]
    # Remove duplicates by using dict.fromkeys to preserve order (set will yield re-ordering)
    colnames = list(dict.fromkeys(colnames))

    # Generate an index column using basin and sitename (WY and run are already in the column names)
    index_col = [peak_dict[k][1] for k in list(peak_dict.keys())]
    # Flatten into a single list
    index_col = h.flatten(index_col)
    # Remove duplicates by using dict.fromkeys to preserve order (set will yield re-ordering)
    index_col = list(dict.fromkeys(index_col))

    # Construct a dataframe of the peak lists from the dict
    # Pull directly, this should generate each row with the basin_run's snotel site peaks
    # The number of columns should be equal to the number of sites in each basin
    peak_val_df = pd.DataFrame([peak_dict[k][3] for k in list(peak_dict.keys())])
    # Transpose this so the rows are now the number of sites in each basin
    peak_val_df = peak_val_df.T

    # now restack for the columns to follow the run_wy, of which there are 8 in all
    # For every 8 columns, extract and stack as new rows into new big dataframe
    for i in range(0, len(peak_val_df.columns), len(colnames)):
        # get this slice
        df_slice = peak_val_df.iloc[:, i:i + len(colnames)]
        # Drop the NaT rows
        df_slice = df_slice.dropna()
        # Specify the colnames for easier merging
        df_slice.columns = colnames
        # Append to a new dataframe
        if i == 0:
            big_peak_val_df = df_slice
        else:
            big_peak_val_df = pd.concat([big_peak_val_df, df_slice], axis=0)
        print(big_peak_val_df.shape)
    # Drop and reset index in place
    big_peak_val_df = big_peak_val_df.reset_index(drop=True)

    # Construct a large dataframe for each basin with SNOTEL sites as index rows and water year and implementation as columns
    # For a total of X snotel site rows by 8 water-year-implementation columns
    big_peak_df = pd.DataFrame(data=big_peak_val_df.values, index=index_col, columns=colnames)
    print(big_peak_df.shape)
    return big_peak_df

big_peak_df = construct_peak_df(isnobal_csvs=isnobal_csvs)

# Convert the whole thing to day of year
big_peak_df_doy = big_peak_df.map(lambda x: x.dayofyear if isinstance(x, pd.Timestamp) else x)
big_peak_df_doy

# Extract corresponding SNOTEL data

In [ ]:
def prep_snotel_meta(basin, state_abbrev='CO', buffer=200,
                poly_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys',
                site_locs_fn='/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL/snotel_sites_32613.json'):
    poly_fn = h.fn_list(poly_dir, f'*{basin}*shp')[0]

    # Locate SNOTEL sites within basin
    found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=site_locs_fn, buffer=buffer)

    # Get site names and site numbers
    sitenames = found_sites['site_name']
    sitenums = found_sites['site_num']
    print(sitenames)
    fixed_names = [site.replace(' ', '').replace('-', '_').replace('#', '').replace('(', '_').replace(')', '') for site in sitenames]
    # prepend basin as well
    fixed_names = [f'{basin}_{site}' for site in fixed_names]
    ST_arr = [state_abbrev] * len(sitenums)
    return sitenames, sitenums, fixed_names, ST_arr

def extract_snotel_peak(sitenums, sitenames, fixed_names, WYs, ST_arr, epsg=32613, return_df=True, return_meta=False, snowvar='SNOWDEPTH'):
    # Prepare your final dataframe of snotel peak with WYs as columns and sites as rows
    snotel_basin_df = pd.DataFrame(index=fixed_names, columns=[f'SNOTEL_wy{WY}' for WY in WYs])

    # Get the SNOTEL data for all input water years so this only happens once
    _, snotel_dfs  = proc.get_snotel(sitenum=sitenums, sitename=sitenames, WY=WYs, ST=ST_arr, epsg=epsg, return_df=return_df, return_meta=return_meta, snowvar=snowvar)

    # From these snotel_dfs, process to pull out the series of interest
    snotel_dfs_vars = [df[f'{snowvar}_m'] for df in snotel_dfs]

    # Combine this list of SNOTEL dataframes into a single dataframe
    temp_snotel_df = pd.concat(snotel_dfs_vars, axis=1)

    # Rename the columns as the sitenames for this basin
    temp_snotel_df.columns = fixed_names

    for WY in WYs:
        # slice up the dataframe based on datetime
        thisWY_df = temp_snotel_df[(temp_snotel_df.index.date >= pd.to_datetime(f'{WY-1}-10-01').date()) & (temp_snotel_df.index.date <= pd.to_datetime(f'{WY}-09-30').date())]
        # Get the peak for this WY for each column in this dataframe with a lambda function
        thisWY_peaks = thisWY_df.apply(lambda x: proc.calc_peak(snow_property=x, verbose=False)[0], axis=0)
        snotel_basin_df[f'SNOTEL_wy{WY}'] = thisWY_peaks
    return snotel_basin_df

In [ ]:
def construct_snotelpeak_df(basins, WYs, epsg=32613, snowvar='SNOWDEPTH', state_abbrev='CO'):
    big_snotel_df_list = list()
    for basin in basins:
        print(basin)
        sitenames, sitenums, fixed_names, ST_arr = prep_snotel_meta(basin=basin, state_abbrev=state_abbrev)
        snotel_basin_df = extract_snotel_peak(sitenums=sitenums, sitenames=sitenames, fixed_names=fixed_names,
                                             WYs=WYs, ST_arr=ST_arr, epsg=epsg, snowvar=snowvar, return_df=True)
        big_snotel_df_list.append(snotel_basin_df)

    big_snotel_df = pd.concat(big_snotel_df_list, axis=0)
    return big_snotel_df

big_snotel_df = construct_snotelpeak_df(basins, WYs)

In [ ]:
big_snotel_df

In [ ]:
# snotel plot
big_snotel_df.plot(marker='_', markersize=10, linestyle='None')
# put legend on the right, outside of the plot
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)

In [ ]:
# Compute DOY, which is easier for plotting (could also just pull that out in the doyflag for calc_peak)
big_snotel_df_doy = big_snotel_df.map(lambda x: x.dayofyear if isinstance(x, pd.Timestamp) else x)
big_snotel_df_doy

# Combine the dataframes

In [ ]:
def make_combo_df(df_list):
    # Construct a large dataframe for each basin with SNOTEL sites as index rows and water year and implementation as columns
    combo_df = pd.concat(df_list, axis=1)

    # Rearrange columns by WY for ease of comparison in dataframe format, e.g., HRRR-SPIReS_wy2021, Baseline_wy2021, SNOTEL_wy2021, etc.
    colnames = list(combo_df.columns)
    print(colnames)
    # Place the SNOTEL_wyYYYY columns after the corresponding HRRR-SPIReS and Baseline columns
    rearranged_colnames = []
    for WY in WYs:
        wy_colnames = [col for col in colnames if f'wy{WY}' in col]
        # Move SNOTEL to the beginning of the list, then Baseline, then HRRR-SPIReS
        wy_colnames = [col for col in wy_colnames if 'SNOTEL' in col] + [col for col in wy_colnames if 'Baseline' in col] + [col for col in wy_colnames if 'HRRR' in col]
        # Use plus equals to not have to flatten the list
        rearranged_colnames += (wy_colnames)
    print(rearranged_colnames)
    # now rearrange the dataframe columns of the combo fig
    combo_df = combo_df[rearranged_colnames]
    return combo_df

combo_df = make_combo_df([big_snotel_df_doy, big_peak_df_doy])

In [ ]:
# Get this basin
def get_basin_df(df, basin):
    # # Plot the df for a single basin
    # this_df = combo_df.iloc[:, 0:3]
    thisbasin_df = df.loc[df.index.str.contains(basin)].T
    # Convert to doy
    thisbasin_df = thisbasin_df.map(lambda x: x.dayofyear if isinstance(x, pd.Timestamp) else x)

    return thisbasin_df
thisbasin_df = get_basin_df(combo_df.iloc[:, 0:3], basins[0])
thisbasin_df.T

In [ ]:
def reorganize_df(df):
    # Set the sitenames as a new column
    df['sitename'] = df.index

    # Create a new index_col
    df['index_col'] = 1

    # and reset the index
    df = df.reset_index(drop=True)

    # Set the index_col as the index
    df = df.set_index('index_col')
    return df

this_run = reorganize_df(thisbasin_df.T)
this_run

In [ ]:
SMALL_SIZE = 14
SMEDIUM_SIZE = 16
MEDIUM_SIZE = 22
BIGGER_SIZE = 24
BIGGEST_SIZE = 28

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGEST_SIZE)  # fontsize of the figure title

In [ ]:
def assess_bar_spacing(stepsize=0.4, spacing=0.1, nruns=3):
    # Get some idea of plotting ytick locs
    ticklist = np.arange(0, len(basins), stepsize)
    for i in range(0, len(basins)):
        basin_tick_start = ticklist[i]
        # Loop through runs
        for r in range(0, nruns):
            tick_cluster = basin_tick_start + spacing * r
            print(tick_cluster)
        print('\n')
assess_bar_spacing()

In [ ]:
colors = ['darkgrey', cmap[0], cmap[1]]
fontsize = SMALL_SIZE
fontstyle ='italic'
color = 'lightgrey'
fontweight = 'semibold'

barlw = 10 #12
baralpha = 1
markersize = 500 #600
markerlw = 1.75 #2.5

# offset between  the basins
offset = np.arange(0, len(basins), 0.4)
run_spacing = 0.1

In [ ]:
# PATCH for VISUALIZATION
# Replace the entries that == 365 and sub in 32
combo_df.where(combo_df != 365, 32, inplace=True)
combo_df

# Plot for looking at

In [ ]:
fig, ax = plt.subplots(1, figsize=(20, 6))
# Loop through the water years
for wyoffset, WY in enumerate(WYs):
    start_date = pd.to_datetime(f'{WY}-03-15')
    end_date = pd.to_datetime(f'{WY}-08-01')
    # Go through each basin
    for bdx, basin in enumerate(basins):
        # Pull the basin for this WY
        thisbasin_df = get_basin_df(combo_df.iloc[:, 0 + 3 * wyoffset : 3 + 3 * wyoffset], basin)
        # Reformat the basin df for plotting
        this_run = reorganize_df(thisbasin_df.T)
        # Go through each of the run names
        for idx, col in enumerate(this_run.columns[:-1]):
            y = this_run.index - 1 + offset[bdx] + run_spacing * idx
            # Could try to color according to sitename col
            # adjust x based on the water year, don't do this to scale
            xoffset = 120 * wyoffset
            ax.scatter(x=this_run[col] + xoffset, y=y, marker='|', s=markersize, linewidth=markerlw,
                    color=colors[idx%3], label=col
                    )
            xmin = this_run[col].min() + xoffset
            xmax = this_run[col].max() + xoffset
            ax.hlines(y=y, xmin=xmin, xmax=xmax, color=colors[idx%3], linestyle='-', linewidth=barlw, alpha=baralpha)

            # Annotate these bars for the very topmost set (last basin, first water year)
            if bdx == len(basins)-1 and wyoffset == 0:
                run_xoffset = 5
                ax.annotate(f'{this_run[col].name.split("_")[0]}',
                            xy=(xmax+run_xoffset, y[0]),
                            xytext=(xmax+run_xoffset, y[0]),
                            ha='left', va='center',
                            color=colors[idx],
                            fontweight=fontweight,
                            fontsize=15,
                            fontstyle=fontstyle)
            # Annotate the basin names at xmin, y and idx==1
            if idx == 1 and wyoffset == 0:
                print(idx, bdx, basin)
                ax.annotate(f'{basin.upper()}', xy=(xmin, y[0]), xytext=(xmin, y[0]), ha='left', va='center',
                            color='k',
                            fontweight=fontweight,
                            fontsize=fontsize, fontstyle=fontstyle)
    if (wyoffset == 0) and (bdx == len(basins)-1):
        miny, maxy = ax.get_ylim()
        print(f'ylims: {miny} – {maxy}')
        # Round to floor using tenths place
        miny = np.floor(miny * 10) / 10
        # Round to ceiling and bump up
        maxy = np.ceil(maxy * 10) / 10
        print(f'ylims: {miny} – {maxy}')
    ax.set_ylim(miny, maxy)
    ymax_labels = maxy - 0.05
    # Add vertical lines at the first of each month between april and june and annotate with Month and Day
    for i in range(start_date.month + 1, end_date.month, 1):
        this_month_date = pd.to_datetime(f'{WY}-{i:02d}-01')
        this_month_doy = this_month_date.dayofyear + xoffset
        if (i == 4) or (i == 7):
            ax.vlines(this_month_doy, ymin=miny, ymax=maxy, color='gray', linestyle='--', linewidth=1, alpha=0.45)
        else:
            ax.vlines(this_month_doy, ymin=miny, ymax=maxy, color='gray', linestyle='--', linewidth=1, alpha=0.25)
        if (i == 4) or (i == 7):
            ax.annotate(f'{this_month_date.month_name()[:3]} 1', xy=(this_month_doy+run_xoffset/2, ymax_labels), xytext=(this_month_doy+run_xoffset/2, ymax_labels),
                        color=color,
                        fontweight=fontweight,
                        fontsize=fontsize, fontstyle=fontstyle)
ax.set_xticks([])
ax.set_yticks([])
# ax.set_xlim(start_date.dayofyear, end_date.dayofyear + xoffset)

# Plot for poster

In [ ]:
fig, ax = plt.subplots(1, figsize=(20*1.07, 6*1.07))
# Loop through the water years
for wyoffset, WY in enumerate(WYs):
    start_date = pd.to_datetime(f'{WY}-03-15')
    end_date = pd.to_datetime(f'{WY}-08-01')
    # Go through each basin
    for bdx, basin in enumerate(basins):
        # Pull the basin for this WY
        thisbasin_df = get_basin_df(combo_df.iloc[:, 0 + 3 * wyoffset : 3 + 3 * wyoffset], basin)
        # Reformat the basin df for plotting
        this_run = reorganize_df(thisbasin_df.T)
        # Go through each of the run names
        for idx, col in enumerate(this_run.columns[:-1]):
            y = this_run.index - 1 + offset[bdx] + run_spacing * idx
            # adjust x based on the water year
            xoffset = 120 * wyoffset
            ax.scatter(x=this_run[col] + xoffset, y=y, marker='|', s=markersize, linewidth=markerlw,
                    color=colors[idx%3], label=col
                    )
            xmin = this_run[col].min() + xoffset
            xmax = this_run[col].max() + xoffset
            ax.hlines(y=y, xmin=xmin, xmax=xmax, color=colors[idx%3], linestyle='-', linewidth=barlw, alpha=baralpha)
    if (wyoffset == 0) and (bdx == len(basins)-1):
        miny, maxy = ax.get_ylim()
        print(f'ylims: {miny} – {maxy}')
        # Round to floor using tenths place
        miny = np.floor(miny * 10) / 10
        # Round to ceiling and bump up
        maxy = np.ceil(maxy * 10) / 10
        print(f'ylims: {miny} – {maxy}')
    ax.set_ylim(miny, maxy)
    ymax_labels = maxy - 0.05
    # Add vertical lines at the first of each month between april and june and annotate with Month and Day
    for i in range(start_date.month + 1, end_date.month, 1):
        this_month_date = pd.to_datetime(f'{WY}-{i:02d}-01')
        this_month_doy = this_month_date.dayofyear + xoffset
        if (i == 4) or (i == 7):
            ax.vlines(this_month_doy, ymin=miny, ymax=maxy, color='gray', linestyle='--', linewidth=1, alpha=0.45)
        else:
            ax.vlines(this_month_doy, ymin=miny, ymax=maxy, color='gray', linestyle='--', linewidth=1, alpha=0.25)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(start_date.dayofyear - xoffset / 4, end_date.dayofyear + xoffset)

In [ ]:
combo_df

In [ ]:
import copy
# Calculate the average difference from SNOTEL_wy2021 for each water year, for each basin
# Add a column as 'difference from SNOTEL' for each water year in each row
# snotel_cols = [col for col in combo_df.columns if 'SNOTEL' in col]
# snotel_cols
# do this on a copy
big_df = copy.deepcopy(combo_df)
# Loop through the water years
for wyoffset, WY in enumerate(WYs):
    # wy_df = big_df.iloc[:, 0 + 3 * wyoffset : 3 + 3 * wyoffset]
    # Calculate the difference from SNOTEL column for each isnobal run for each year
    big_df[f'Baseline_diff_wy{WY}'] = big_df[f'Baseline_wy{WY}'] - big_df[f'SNOTEL_wy{WY}']
    big_df[f'HS_diff_wy{WY}'] = big_df[f'HRRR-SPIReS_wy{WY}'] - big_df[f'SNOTEL_wy{WY}']

In [ ]:
# Get basin dfs
basin_dfs = []
for basin in basins:
    print(basin)
    thisbasin_df = get_basin_df(big_df, basin)
    basin_dfs.append(thisbasin_df.T)

In [ ]:
# Change legend font to 12
basin_dfs[0].describe()
# .plot(figsize=(12, 4))
# plt.legend(fontsize=12, bbox_to_anchor=(1, 1), loc='upper left')

In [ ]:
basin_dfs[1].describe()

In [ ]:
basin_dfs[2].describe()